### Agentic RAG

Agentic RAG is an advanced form of RAG (Retrieval-Augmented Generation) where an AI agent actively decides how to retrieve, verify, and use information instead of following a fixed retrieval pipeline.

In [ ]:
import os
from  typing import List , Annotated
from pydantic import BaseModel

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
llm.invoke("Hello")


In [ ]:
### 1. Document Processing

urls=[
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/"
]

loaders = [WebBaseLoader(url) for url in urls]

docs=[]

for loader in loaders:
    docs.extend(loader.load())

docs

In [ ]:
## 2. Recursive character txt splitter as vector store

splitter =RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(split_docs, embedding)
retriever= vectorstore.as_retriever()

retriever.invoke("what is reward function") # test the retriever

In [ ]:
# 3. Define State of the Graph 

class RAGState(BaseModel):
    question:str
    retrieved_docs:List[Document]=[]
    answer:str=""

In [ ]:
# 4. Define the nodes

def retrieve_docs(state:RAGState)->RAGState:
    docs=retriever.invoke(state.question)
    return RAGState(question=state.question,retrieved_docs=docs)


def generate_answer(state:RAGState)->RAGState:

    context= "\n\n".join(doc.page_content  for doc in docs)
    prompt= f"Answer the question based on the context. \n\nContext:\n{context} \n\n Question:\n{state.question}"
    response = llm.invoke(prompt)
    return RAGState(question=state.question, retrieved_docs=docs, answer=response.content)


In [ ]:
# 5. Build LangGraph
from IPython.display import Image, display


builder = StateGraph(RAGState)

builder.add_node("retrieve_docs" , retrieve_docs)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "retrieve_docs")
builder.add_edge( "retrieve_docs", "generate_answer")
builder.add_edge( "generate_answer", END)

graph_builder = builder.compile()

# View the graph
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
# Invoke the graph

if __name__=="__main__":
    user_question="what is Reward tampering?"
    initial_state = RAGState(question=user_question)
    final_state= graph_builder.invoke(initial_state)

    print("Final Answer", final_state['answer'])